In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import re

def split_sentences(text: str):
    # simple, fast, no external models
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 0]


model = SentenceTransformer("BAAI/bge-base-en")

MAX_TOKENS = 350     # sweet spot for bge-base
SIM_THRESHOLD = 0.75 # semantic continuity threshold


c:\Users\shravan\Documents\OMNI_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 425.17it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
from nltk.tokenize import sent_tokenize
from llama_index.core.schema import TextNode

def semantic_chunk_pdf(doc):
    sentences = split_sentences(doc.text)


    sentence_embeddings = model.encode(sentences, normalize_embeddings=True)

    chunks = []
    current_chunk = [sentences[0]]
    current_emb = sentence_embeddings[0]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(
            current_emb.reshape(1, -1),
            sentence_embeddings[i].reshape(1, -1)
        )[0][0]

        prospective_text = " ".join(current_chunk + [sentences[i]])

        if sim > SIM_THRESHOLD and len(prospective_text.split()) < MAX_TOKENS:
            current_chunk.append(sentences[i])
            current_emb = np.mean(
                model.encode(current_chunk, normalize_embeddings=True),
                axis=0
            )
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
            current_emb = sentence_embeddings[i]

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    nodes = []
    for chunk in chunks:
        node = TextNode(
            text=chunk,
            metadata=doc.metadata.copy()
        )
        nodes.append(node)

    return nodes


In [3]:
import pickle

with open("../data/processed/all_documents.pkl", "rb") as f:
    pdf_docs = pickle.load(f)

print(f"Loaded {len(pdf_docs)} documents")


Loaded 15 documents


In [4]:
assert isinstance(pdf_docs, list)
assert len(pdf_docs) > 0

assert all(
    d.metadata.get("source_type") == "pdf"
    for d in pdf_docs
), "Non-PDF document detected — ingestion drift"


In [5]:
pdf_nodes = []

for doc in pdf_docs:
    pdf_nodes.extend(semantic_chunk_pdf(doc))

print("PDF semantic chunks:", len(pdf_nodes))


PDF semantic chunks: 23


In [6]:
all_nodes = []

for idx, node in enumerate(pdf_nodes ):
    node.metadata["chunk_id"] = idx
    node.metadata["embedding_model"] = "bge-base-en"
    node.metadata["chunk_strategy"] = (
        "semantic_sentence_merge"
        if node in pdf_nodes else "row_atomic"
    )
    all_nodes.append(node)

print("Total chunks:", len(all_nodes))


Total chunks: 23


In [7]:
import pickle

with open("../data/processed/chunks.pkl", "wb") as f:
    pickle.dump(all_nodes, f)


In [8]:
for i in range(3):
    print("=" * 80)
    print(all_nodes[i].text)
    print(all_nodes[i].metadata)


1. What is the core idea behind Bliss-LLM? Bliss-LLM is a hardware-aware auto-tuning engine that optimizes LLM inference parameters like
temperature, top_p, and precision using Bayesian optimization to minimize latency while
maintaining output quality. 2. How is Bliss-LLM different from fine-tuning? Fine-tuning changes model weights to improve accuracy; auto-tuning changes runtime parameters
to improve speed and efficiency without retraining. 3. What parameters are optimized in Bliss-LLM? Temperature, top_p, context_length, and precision (int8/fp16) are tuned to achieve optimal speed
and quality. 4. Why use Bayesian Optimization instead of grid or random search? Bayesian optimization is sample-efficient and uses probabilistic models to intelligently explore the
parameter space, reducing the number of runs needed. 5. How is latency measured during optimization? Each configuration is tested by calling Ollama API, recording start and end time, and returning
latency in seconds. 6. What hap